In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install ../databricks_helpers
dbutils.library.restartPython()

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS raw_data;
SELECT current_catalog(), current_schema();

## Idempotent AutoDownloading of data from BTS site

In [0]:
import importlib
import fetch_bts_data
import os

#for reloading modules
#importlib.reload(fetch_bts_data)

years = [2022, 2023, 2024, 2025]
bts_out_dir = f"{dbh.volume_directory()}/bts_data"
bts_static_dir = f"{bts_out_dir}/static"
os.makedirs(bts_out_dir, exist_ok=True)

# cleaning directory before adding files
# dbutils.fs.rm(bts_out_dir, True)
[fetch_bts_data.fetch_data_yearly(y, bts_out_dir) for y in years]

In [0]:
len([file.name for file in dbutils.fs.ls(bts_out_dir)])

## Converting batch to daily data

In [0]:
#spark.read.format("csv").option("header", "true").load(f"file://{paths_in_this_folder}/airline_data_2022_august.csv").display()

spark.read.format("csv").option("header", "true").load(f"{bts_out_dir}/airline_data_2025_january.csv").display()


In [0]:
input_path = f"{dbh.volume_directory()}/bts_data/*.csv"
combined_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(input_path)

In [0]:
#Converting using pandas api
import pyspark.pandas as pd
input_path = f"{dbh.volume_directory()}/bts_data/*.csv"

combined_df = pd.read_csv(input_path)
display(combined_df)

In [0]:
from pyspark.sql.functions import try_to_date,to_timestamp, date_format,year,month,dayofmonth

output_path = f"{dbh.volume_directory()}/daily_bts_data"
os.makedirs(output_path, exist_ok=True)

combined_df = combined_df.withColumn("FL_DATE", to_timestamp("FL_DATE", "M/d/yyyy h:mm:ss a"))\
 .withColumn("year", year("FL_DATE"))\
 .withColumn("month", month("FL_DATE"))\
 .withColumn("day", dayofmonth("FL_DATE"))



In [0]:
display(combined_df)

In [0]:
combined_df.write.format("parquet")\
    .option("header", "true")\
    .partitionBy("year", "month", "day")\
    .mode("overwrite")\
    .save(output_path)